# **1. 멀티 모달**
- 사람은 사진만 보거나 글만 긁지 않음 -> 눈으로 보고, 말을 듣고, 글을 읽고, 상황을 함께 해석함
- 멀티모달도 위와 비슷하게 서로 다른 형태의 정보를 함께 다룸
    - 이미지: 픽셀, 사물, 색, 위치, 장면
    - 텍스트: 단어, 문장, 의미, 질문
    - 오디오: 음성, 소리, 음악
    - 비디오: 이미지의 시간적 변화 + 경우에 따라 오디오와 함께

> Multimodal Learning은 서로 다른 여러 모달리티의 데이터를 함께 학습하는 방식입니다. 예를 들어 이미지와 텍스트를 동시에 입력으로 받아 장면을 설명하거나, 영상과 음성을 함께 활용해 감정을 인식하는 모델이 이에 해당합니다. 각 모달리티가 가진 상호보완적인 정보를 결합함으로써 단일 모달리티보다 더 풍부한 표현과 높은 성능을 기대할 수 있습니다. 최근에는 비전과 언어를 결합한 모델들이 대표적인 성공 사례로 주목받고 있습니다.

# **2. CLIP**

CLIP은 이미지와 텍스트를 같은 의미 공간으로 정렬하기 위해 대비 학습(contrastive learning)을 사용하는 대표적인 Pretrained Vision–Language 모델입니다. 
<br>이미지 인코더와 텍스트 인코더를 각각 학습시킨 뒤, 올바른 이미지–텍스트 쌍은 임베딩 공간에서 가깝게, 관련 없는 쌍은 멀어지도록 학습함으로써 두 모달리티 간 의미적 대응 관계를 형성합니다. 
<br>이 구조 덕분에 별도의 태스크별 학습 없이도 텍스트로 정의한 클래스 설명만으로 이미지를 분류하는 zero-shot 분류가 가능하며, 
<br>이미지–텍스트 검색(Text-to-Image, Image-to-Text Retrieval)에 특히 강점을 보입니다. 다만 생성 능력은 없고, 주로 “이해와 정렬”에 초점이 맞춰진 모델이라는 특징을 가집니다
<br>([논문](https://arxiv.org/pdf/2103.00020)) https://arxiv.org/pdf/2103.00020

### 표현 정렬과 생성의 차이
- Dual encoder(CLIP): 이미지와 문장을 독립적으로 벡터화하고 유사도를 비교. 빠른 검색/zero-shot 분류에 적합
- Encoder-decoder(BLIP): 이미지 특징을 참고해 문장을 토큰 단위로 생성
- Ground detection(Grounding DINO): 텍스트 표현을 조건으로 이미지 속 위치(박스)를 찾음
- VLM/멀티모달 LLM: 비전 인코더와 언어 모델을 연결하여 이미지 기반 질의응답/추론/대화를 수행할 수 있음. 모델별 성능과 지원 모달리티는 다름


- 인코더: 원본 데이터를 모델이 계산하기 좋은 숫자 특징을 바꾸는 부분
    - 예) 이미지 인코더 사진을, 텍스트 인코더는 문장을 받아 벡터로 바꿈
- 임베딩: 데이터의 의미를 숫자 배열(벡터)로 표현한 것. 비슷한 의미를 가진 데이터가 한 방향에 놓이도록 학습함
- 프로젝션: 이미지 인코더와 텍스트 인코더가 만든 서로 다른 특징을 같은 차원의 공통 공간으로 옮기는 변환
- 유사도: 두 벡터가 의미적으로 얼마나 가까운지 나타내는 값


### 대비 학습(contrastive learning)

```
한 배치에 이미지 4장과 각 이미지의 설명 4개가 있음
```
|        | 문장1 | 문장2 | 문장3 | 문장4 |
|--------|-------|-------|-------|-------|
| 이미지1 | 높음  | 낮음  | 낮음  | 낮음  |
| 이미지2 | 낮음  | 높음  | 낮음  | 낮음  |
| 이미지3 | 낮음  | 낮음  | 높음  | 낮음  |
| 이미지4 | 낮음  | 낮음  | 낮음  | 높음  |


> 정답 쌍은 대각선. 학습은 대각선 점수는 높이고, 나머지를 낮추는 방향으로 진행. CLIP은 이미지 -> 텍스트 검색, 이미지 검색을 모두 수행할 수 있게 학습


```
고양이 사진 <>  "a photo of a cat" (가깝게)
고양이 사진 <>  "a photo of a banana" (멀게)
고양이 사진 <>  "a photo of a TV" (멀게)

매우 많은 이미지-텍스트 쌍으로 학습을 반복하면 모델은 단순 클래스 번호가 아니라 시각적 특징과 언어 의미의 관계를 학습하게 됨
```

### Zero-Shot 분류
1. 후보 클래스 이름을 문장으로 만듦 예) "a photo of a cat"
2. 후보 문장들의 텍스트 임베딩을 만듦
3. 입력 이미지 임베딩과 각 텍스트 임베딩의 유사도를 비교함
4. 가장 유사한 텍스트를 예측 클래스로 사용함

> Shot은 새로운 작업에서 제공하는 학습 예제 수
<br> 예) Zero-shot은 해당 작업을 위해 정답 예제를 새로 주고 재학습하지 않은 상태에서 수행한다는 뜻
<br> CLIP에서는 클래스 이름을 자연어 prompt로 만들어 비교하기 때문에 별도의 분류기 학습 없이 새로운 후보를 비교할 수 있음